# Calculate the spatial gradient of a field on the mesh.

In [2]:
# Test evaluate ufl expression

from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystemName = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystemName=coordinateSystemName, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

space = getFELiCSSpace(mesh, order=2, dim=1)

from FELiCS.Fields.Field import Field

Field1_init = Field(space, mesh=mesh)
Field1_init.importH5("function_values_sine")
Field1 = Field(space, mesh=mesh)
Field1.importH5("function_values_sine")

from ufl import TestFunction, dx, conj
import dolfinx.fem
v = TestFunction(space)
expression = Field1.function * conj(v) * dx

Field1.evaluateUflExpression(expression)    

result = Field1.getCoefficientArray()-Field1_init.getCoefficientArray()
print(sum(result))



Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


(-3.165490124704671e-15+0j)


In [2]:
# Second test: Evaluate a gradient

from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName         = "./square_mesh.msh"
coordinateSystemName = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystemName=coordinateSystemName, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

space_scalar = getFELiCSSpace(mesh, order=2, dim=1)
space_vec    = getFELiCSSpace(mesh, order=2, dim=2)

from FELiCS.Fields.Field import Field

Field1 = Field(space_scalar, mesh=mesh)
Field1.importH5("function_values_sine")
Field_grad = Field(space_vec, mesh=mesh)
Field_grad_sol = Field(space_vec, mesh=mesh)
Field_grad_sol.importH5("grad_solution")

from ufl import TestFunction, dx, conj
from FELiCS.Misc.tensorUtils import iGrad, iConj, iDot, Tensor
import dolfinx.fem
J_hat = mesh.coordinateSystem.J_hat
v = TestFunction(space_vec)
v_tens = Tensor(v, CoordSys=mesh.coordinateSystem, hasSpectralDimension=False)
expression = iDot(iGrad(Field1.getTensor()), iConj(v_tens)).ufl_tens * J_hat* dx

Field_grad.evaluateUflExpression(expression)    

print(sum(Field_grad.getCoefficientArray() - Field_grad_sol.getCoefficientArray()))
print(sum(Field_grad_sol.getCoefficientArray()))
print(sum(Field_grad.getCoefficientArray()))

Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


(-0.1487525173112279+0j)
(41.78913098529694+0j)
(41.64037846799268+0j)


In [8]:
# Third test: Evaluate gradient with cylindrical coordinates
import numpy as np
from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystemName = "Cylindrical"
m = 3

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystemName=coordinateSystemName, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

space_scalar = getFELiCSSpace(mesh, order=2, dim=1)
space_vec    = getFELiCSSpace(mesh, order=2, dim=3)

from FELiCS.Fields.Field import Field

Field1 = Field(space_scalar, mesh=mesh, m=3)
Field1.importH5("function_values_sine")
Field_grad = Field(space_vec, mesh=mesh)
Field_grad_sol = Field(space_vec, mesh=mesh)
Field_grad_sol.importH5("grad_solution_cylSpectral_m=3")

from ufl import TestFunction, dx, conj
from FELiCS.Misc.tensorUtils import iGrad, iConj, iDot, Tensor
import dolfinx.fem
J_hat = mesh.coordinateSystem.J_hat
v = TestFunction(space_vec)
v_tens = Tensor(v, CoordSys=mesh.coordinateSystem, m = 3, hasSpectralDimension=True)
expression = iDot(iGrad(Field1.getTensor()), iConj(v_tens)).ufl_tens * J_hat* dx

Field_grad.evaluateUflTensorExpression(expression)    

print(np.linalg.norm(Field_grad.getCoefficientArray() - Field_grad_sol.getCoefficientArray()))
print(np.linalg.norm(Field_grad_sol.getCoefficientArray()))
print(np.linalg.norm(Field_grad.getCoefficientArray()))

print(np.max(Field_grad_sol.getCoefficientArray()))
print(np.max(Field_grad.getCoefficientArray()))



Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


3783.773750795523
789.8874378862741
3700.4227200259206
(6.283185307179586+0j)
(6.364538026936767+0j)


In [7]:
import numpy as np

def test_function_2(x):
    """Sine wave function: f(x,y) = sin(2*pi*x) * cos(2*pi*y)"""
    return np.sin(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1])

def dtest_dx(x):
    """Derivative of the sine wave function with respect to x: df/dx = 2*pi*cos(2*pi*x)*cos(2*pi*y)"""
    return 2 * np.pi * np.cos(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1])

def dtest_dy(x):
    """Derivative of the sine wave function with respect to y: df/dy = -2*pi*sin(2*pi*x)*sin(2*pi*y)"""
    return -2 * np.pi * np.sin(2 * np.pi * x[0]) * np.sin(2 * np.pi * x[1])

def dtest_dx_spectralD(x):
    """Derivative of the sine wave function with respect to x: df/dx = 2*pi*cos(2*pi*x)*cos(2*pi*y)"""
    return 2 * np.pi * np.cos(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1]) 

def dtest_dr_spectralD(x):
    """Derivative of the sine wave function with respect to y: df/dy = -2*pi*sin(2*pi*x)*sin(2*pi*y)"""
    return -2 * np.pi * np.sin(2 * np.pi * x[0]) * np.sin(2 * np.pi * x[1])
    
def dtest_dt_spectralD(x,m):
    """Derivative of the sine wave function with respect to y: df/dy = -2*pi*sin(2*pi*x)*sin(2*pi*y)"""
    if x[1]<1.e-10:
        factor = 0.
    else:
        factor = 1./x[1]        
    return factor * np.sin(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1]) * 1j * m 


# Get coordinates of all DOFs in the P2 space
from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace
space = getFELiCSSpace(mesh, order=2, dim=1)
dof_coordinates = space.tabulate_dof_coordinates()
print(f"Number of DOFs: {len(dof_coordinates)}")
print(f"DOF coordinates shape: {dof_coordinates.shape}")

# Calculate function values at DOF coordinates
function_values_1 = np.zeros(len(dof_coordinates))
function_values_2 = np.zeros(len(dof_coordinates))

for i, coord in enumerate(dof_coordinates):
    function_values_1[i] = dtest_dx(coord)
    function_values_2[i] = dtest_dy(coord)

values = np.hstack((function_values_1, function_values_2))
np.save("grad_solution.npy", values)

# Calculate function values for cylindrical coordinate system
function_values_1 = np.zeros(len(dof_coordinates),dtype=complex)
function_values_2 = np.zeros(len(dof_coordinates),dtype=complex)
function_values_3 = np.zeros(len(dof_coordinates),dtype=complex)

m = 3

for i, coord in enumerate(dof_coordinates):
    function_values_1[i] = dtest_dx_spectralD(coord)
    function_values_2[i] = dtest_dr_spectralD(coord)
    function_values_3[i] = dtest_dt_spectralD(coord, m)


values = np.hstack((function_values_1, function_values_2, function_values_3))
#values = np.where(np.isnan(values)==True, 0., values)

np.save("grad_solution_cylSpectral_m=3.npy", values)




Number of DOFs: 1969
DOF coordinates shape: (1969, 3)
